# 02 · The Good

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/splevine/clustering-good-bad-beautiful/blob/main/notebooks/01_the_good.ipynb)

**A modern clustering pipeline:** UMAP → HDBSCAN → BERTopic, run on 5,000 movie overviews.

Noise becomes a first-class citizen; cluster counts come from the data, not a guess; and every cluster gets an interpretable label.

## Setup

In [ ]:
# Colab: uncomment to install
# !pip install -q sentence-transformers umap-learn hdbscan bertopic pandas pyarrow matplotlib

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.rcParams["figure.dpi"] = 110

## Load movies and embeddings

We cache the overview embeddings here so `02_the_bad` and `03_the_beautiful` can reuse them instantly. First run takes ~30 sec on CPU.

In [ ]:
from sentence_transformers import SentenceTransformer

movies = pd.read_parquet("../data/movies.parquet")
movies = movies[movies["overview"].str.len() > 30].reset_index(drop=True)
overviews = movies["overview"].tolist()

EMB_DIR = Path("../embeddings")
EMB_DIR.mkdir(exist_ok=True)
emb_path = EMB_DIR / "overviews_minilm.npy"

if emb_path.exists():
    X = np.load(emb_path)
else:
    model = SentenceTransformer("all-MiniLM-L6-v2")
    X = model.encode(overviews, show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=True)
    np.save(emb_path, X)

print(f"{len(movies):,} movies, embeddings shape {X.shape}")

## 1. UMAP — structure-preserving reduction

We build **two** UMAP projections:
- `umap_cluster` at **5 dimensions** for HDBSCAN to run on. Higher than 2, so clustering has room to breathe; lower than 384, so density becomes meaningful.
- `umap_viz` at **2 dimensions** for plotting.

Key knobs: `n_neighbors` (local vs. global structure), `min_dist` (cluster tightness), `metric` (cosine for unit-normalized embeddings).

In [ ]:
from umap import UMAP

umap_cluster = UMAP(
    n_components=5, n_neighbors=15, min_dist=0.0, metric="cosine", random_state=0
).fit_transform(X)
umap_viz = UMAP(
    n_components=2, n_neighbors=15, min_dist=0.1, metric="cosine", random_state=0
).fit_transform(X)
print(f"cluster space: {umap_cluster.shape}  |  viz space: {umap_viz.shape}")

## 2. HDBSCAN — density-based clusters with noise

No `k` to guess. `min_cluster_size` sets the smallest group we're willing to call a cluster — everything smaller becomes noise (label `-1`). That's the point: noise is a valid answer.

In [ ]:
import hdbscan

clusterer = hdbscan.HDBSCAN(min_cluster_size=20, min_samples=5, cluster_selection_method="eom")
labels = clusterer.fit_predict(umap_cluster)
movies["cluster"] = labels

n_clusters = (labels.max() + 1) if labels.max() >= 0 else 0
n_noise = int((labels == -1).sum())
print(f"{n_clusters} clusters, {n_noise} noise ({n_noise / len(labels):.1%})")
print("\nLargest 10 clusters:")
print(pd.Series(labels).value_counts().head(11).to_string())

## 3. BERTopic — interpretable labels

BERTopic wraps embedding + reduction + clustering + labeling in one object. We hand it the models we already built so the labels match the clustering you just saw.

In [ ]:
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(stop_words="english", min_df=2, ngram_range=(1, 2))

topic_model = BERTopic(
    embedding_model="all-MiniLM-L6-v2",
    umap_model=UMAP(n_components=5, n_neighbors=15, min_dist=0.0, metric="cosine", random_state=0),
    hdbscan_model=hdbscan.HDBSCAN(min_cluster_size=20, min_samples=5, cluster_selection_method="eom", prediction_data=True),
    vectorizer_model=vectorizer,
    verbose=False,
)
topics, _ = topic_model.fit_transform(overviews, embeddings=X)
movies["topic"] = topics

topic_info = topic_model.get_topic_info()
print(topic_info.head(12).to_string(index=False))

## 4. Eyeball a few topics

For each of the top clusters, pull the top terms (c-TF-IDF) and the most representative movies. This is the human sanity check: does the label match the movies?

In [ ]:
def describe_topic(topic_id, n_movies=5, n_terms=6):
    terms = [t for t, _ in topic_model.get_topic(topic_id)[:n_terms]]
    in_topic = movies[movies["topic"] == topic_id].sort_values("vote_count", ascending=False)
    titles = in_topic["title"].head(n_movies).tolist()
    return terms, titles

top_topics = [t for t in topic_info["Topic"].head(10) if t != -1][:6]
for t in top_topics:
    terms, titles = describe_topic(t)
    print(f"Topic {t:>2}  |  terms: {', '.join(terms)}")
    print(f"            examples: {', '.join(titles)}\n")

## 5. Validate against genres

We have TMDB genre tags on each movie — noisy but useful as a crude ground truth. If our unsupervised clusters align with genres, that's real evidence that clustering found signal, not just story.

In [ ]:
movies["primary_genre"] = movies["genres"].map(lambda gs: gs[0] if len(gs) else "Unknown")

# Use the HDBSCAN cluster column (already set on movies) for validation.
xtab = pd.crosstab(movies["cluster"], movies["primary_genre"])
# For each cluster, show what fraction of it is the top genre.
cluster_purity = xtab.div(xtab.sum(axis=1), axis=0).max(axis=1)
print("Dominant-genre share per cluster (higher = more genre-pure):")
print(cluster_purity.drop(-1, errors="ignore").sort_values(ascending=False).head(10).round(2).to_string())
print("\nOverall:")
print(f"  mean dominant-genre share (non-noise clusters): {cluster_purity.drop(-1, errors='ignore').mean():.2f}")
print(f"  baseline (most common genre globally):          {movies['primary_genre'].value_counts(normalize=True).iloc[0]:.2f}")

If the mean dominant-genre share is meaningfully above the baseline (the frequency of the most common genre overall), our clusters are picking up genre signal — not perfectly, but beyond chance. The movies that *don't* land in their primary genre's cluster are often the interesting ones — genre benders.

## 6. Visualize

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

noise_mask = labels == -1
ax.scatter(umap_viz[noise_mask, 0], umap_viz[noise_mask, 1], c="lightgray", s=3, alpha=0.3, label=f"noise ({noise_mask.sum()})")

signal_labels = labels[~noise_mask]
ax.scatter(umap_viz[~noise_mask, 0], umap_viz[~noise_mask, 1], c=signal_labels, cmap="tab20", s=4, alpha=0.8)
ax.set_title(f"HDBSCAN on UMAP(5-D) movie overviews · {n_clusters} clusters, {n_noise} noise")
ax.set_xticks([]); ax.set_yticks([])
ax.legend(loc="lower left", fontsize=9)
plt.tight_layout(); plt.show()

## Pitfalls, even with a good pipeline

- **Over-interpreting tiny clusters.** A cluster of 22 movies might be meaningful or might be a UMAP artifact.
- **`min_cluster_size` matters.** Sweep it (try 10, 20, 50, 100) and see which cluster structure survives. Structure that survives across settings is more trustworthy.
- **Noise is informative.** The 10–20% labeled `-1` often contains the most interesting movies — genre-benders, cross-language hits, experimental films. Don't silently drop it.

Next: [`03_the_beautiful.ipynb`](03_the_beautiful.ipynb) — swap text overviews for movie posters and see what a completely different modality brings.